# Bucketing (Co-partitioned Joins)

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

Bucketing requires **managed tables** (`saveAsTable`, not a bare `.parquet()` path write) -- Spark reads back the bucketing metadata (`BucketSpec`) from the catalog to skip the shuffle at join time.

In [ ]:
import shutil
import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from driver.playbook import checkpoint

# Managed tables (saveAsTable, required for bucketBy) default to a *cwd-relative* spark-warehouse/
# directory -- and a notebook kernel's cwd is the notebook's own directory, not /workspace. Left
# unset, this topic's tables would land inside content/bucketing/ itself (verified against a real
# cluster while building this notebook -- exactly that happened). Point the warehouse at scratch/
# instead, matching every other topic's generated-data convention.
WAREHOUSE_DIR = "/workspace/scratch/bucketing/warehouse"

spark = SparkSession.builder.appName("bucketing") \
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR) \
    .getOrCreate()

# Disable broadcast joins entirely -- this topic is about bucketing avoiding a *shuffle*, and a small
# enough table would otherwise just get broadcast instead, masking the thing being taught.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

# Spark 3.1+ can auto-disable the bucketed-read optimization when its own heuristic decides a plain
# shuffle would be cheaper (spark.sql.sources.bucketing.autoBucketedScan.enabled, default true). Force it
# off so this exercise deterministically exercises the co-partitioned-join path.
spark.conf.set("spark.sql.sources.bucketing.autoBucketedScan.enabled", "false")

# DROP TABLE IF EXISTS alone is NOT enough to make this notebook safely re-runnable against a
# freshly-respawned cluster (issue #14, found in Phase 2 acceptance validation): the in-memory Hive
# catalog that DROP TABLE acts on does not survive a container teardown, but WAREHOUSE_DIR is a
# host-side bind-mounted path that DOES survive -- so a second run against a fresh cluster hit
# 'SparkRuntimeException: [LOCATION_ALREADY_EXISTS] ... bucketed_a already exists' because the prior
# run's on-disk files were still there even though the catalog that DROP TABLE checks was reset.
# Remove the on-disk directories too, for the same tables, so a fresh cluster + fresh catalog +
# leftover-but-now-cleaned files always starts from a truly clean slate with no manual host cleanup.
for tbl in ("bucketed_a", "bucketed_b", "bucketed_c_mismatched"):
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    shutil.rmtree(f"{WAREHOUSE_DIR}/{tbl}", ignore_errors=True)

## 1. Write two tables bucketed on the same key and bucket count

Both `bucketed_a` and `bucketed_b` are bucketed on `id` into the same number of buckets (8). Rows sharing an `id` land in the same bucket file in both tables -- that's what lets Spark skip the shuffle when it joins them.

In [ ]:
NUM_ROWS = 800_000
NUM_BUCKETS = 8

a_rdd = spark.sparkContext.parallelize(range(NUM_ROWS), 16).map(
    lambda i: Row(id=i, value_a=float(i % 1000))
)
spark.createDataFrame(a_rdd).write.mode("overwrite") \
    .bucketBy(NUM_BUCKETS, "id").sortBy("id").saveAsTable("bucketed_a")

b_rdd = spark.sparkContext.parallelize(range(NUM_ROWS), 16).map(
    lambda i: Row(id=i, value_b=float((i * 7) % 1000))
)
spark.createDataFrame(b_rdd).write.mode("overwrite") \
    .bucketBy(NUM_BUCKETS, "id").sortBy("id").saveAsTable("bucketed_b")

print("Wrote bucketed_a and bucketed_b, both bucketed on id into", NUM_BUCKETS, "buckets.")

## 2. Join on the bucketing key -- hypothesize before running

**Hypothesis:** will this join's plan contain an `Exchange` node? Why or why not?

In [ ]:
co_partitioned_join = spark.table("bucketed_a").join(spark.table("bucketed_b"), on="id", how="inner")
co_partitioned_join.count()
co_partitioned_join.explain(mode="formatted")

In [ ]:
checkpoint(co_partitioned_join, topic="bucketing")
print("Checkpoint written -- click 'Reveal self-check' on the topic page.")

## 3. Contrast case: mismatched bucket count still shuffles

`bucketed_c_mismatched` is bucketed on the *same key* but into a *different* number of buckets (4, not 8). Same key, same optimization intent, but Spark cannot guarantee co-partitioning across a differing bucket count, so this join shuffles anyway -- bucketing is not automatically load-bearing, the bucket spec has to actually line up.

**Hypothesis:** does this plan show a `SortMergeJoin` with or without a preceding `Exchange`?

In [ ]:
c_rdd = spark.sparkContext.parallelize(range(NUM_ROWS), 16).map(
    lambda i: Row(id=i, value_c=float((i * 3) % 1000))
)
spark.createDataFrame(c_rdd).write.mode("overwrite") \
    .bucketBy(4, "id").sortBy("id").saveAsTable("bucketed_c_mismatched")

mismatched_join = spark.table("bucketed_a").join(
    spark.table("bucketed_c_mismatched"), on="id", how="inner"
)
mismatched_join.count()
mismatched_join.explain(mode="formatted")

In [ ]:
checkpoint(mismatched_join, topic="bucketing")
print("Checkpoint written -- Reveal again: this plan's SortMergeJoin should now be labeled distinctly "
      "from the no-shuffle case above (US-2.4), not collapsed into the same concept.")

## 4. Clean up

These are managed tables in the cluster's default warehouse -- drop them (catalog *and* on-disk files, see the setup cell's note on issue #14) if you want a clean slate for a re-run. Not required: the setup cell already does this automatically at the top of every run.

In [ ]:
# Uncomment to clean up:
# for tbl in ("bucketed_a", "bucketed_b", "bucketed_c_mismatched"):
#     spark.sql(f"DROP TABLE IF EXISTS {tbl}")
#     shutil.rmtree(f"{WAREHOUSE_DIR}/{tbl}", ignore_errors=True)